# Zomato Analytics — Silver Layer (Cleansing & Transformation)

**Purpose**: Cleanse, deduplicate, standardize, and enrich Bronze data into a
conformed, analytics-ready Silver layer. Applies business rules, data-type casting,
null handling, and referential integrity checks.

**Unity Catalog**: `zomato_analytics.silver.*`

**Input**: `zomato_analytics.bronze.*`
**Output**: `zomato_analytics.silver.*`

**Exercise 5**: `silver.delivery_partners` is loaded from Lakebase CDC — `MERGE` (incremental) or rebuild (full).

**SLA**: Triggered after Bronze ingestion completes | Max latency: 30 min

## Exercise 5 — Lakebase CDC into Silver: how this solution works
```
  ┌────────────────────────────────────────┐
  │ LAKEBASE  (Postgres OLTP database)     │
  │ zomato_ops.delivery_partners           │  <── partner app writes here
  │   │  (a trigger copies every change)   │
  │   └─> zomato_ops.cdc_changelog         │  one row per INSERT / UPDATE / DELETE
  └────────────────────┬───────────────────┘
                       │  ① BRONZE: copy only the NEW changes (full load = whole-table snapshot)
                       ▼
  ┌────────────────────────────────────────┐
  │ bronze.lakebase_delivery_partners_cdc  │  every change received = history
  └────────────────────┬───────────────────┘
                       │  ② SILVER: newest change per partner wins → MERGE  <── THIS NOTEBOOK
                       ▼
  ┌────────────────────────────────────────┐
  │ silver.delivery_partners               │  1 row per partner = how it looks NOW
  └────────────────────┬───────────────────┘
                       │  ③ GOLD: read what changed in Silver → add a new version
                       ▼
  ┌────────────────────────────────────────┐
  │ gold.dim_delivery_partners             │  1 row per partner per version (SCD Type 2)
  └────────────────────────────────────────┘
```

### Silver algorithm (section 8A)

```
1. Pick rows     FULL        : all Bronze rows
                 INCREMENTAL : Bronze rows with txid > highest txid already in Silver (our bookmark)
2. Newest wins   if a partner changed several times, keep only the change with the highest change number
3. Clean         names, phone, email, rating (1–5), partner tier …
                 a DELETE becomes  is_deleted = true  (the row is kept — "soft delete")
4. Write         FULL        : overwrite Silver
                 INCREMENTAL : MERGE — insert new partners, update known partners
                               ONLY IF the change is newer than what Silver already has
```

**Example — one incremental run**

| Bronze rows since last run | | Silver after the MERGE |
|---|---|---|
| `DP_01` UPDATE city = Pune (change #501) | | |
| `DP_01` UPDATE city = Mumbai (change #507) | → | `DP_01` city = Mumbai *(newest wins)* |
| `DP_02` DELETE (change #503) | → | `DP_02` kept, `is_deleted = true` |
| `DP_03` INSERT (change #509) | → | `DP_03` new row |

Why soft delete? Old deliveries were made by partners who may have left — they must still find their partner.
Why "only if newer"? Running the same batch twice (retry, replay) can never overwrite newer data → **safe to re-run**.
Silver has **Change Data Feed** switched on, so Gold can read only the partners that changed.

**Two ways to run it** — widget / job parameter `lakebase_load_type` (use the same value in Bronze, Silver and Gold):

| Layer | `full` | `incremental` (default) |
|---|---|---|
| Bronze | snapshot of the whole Lakebase table → **overwrite** | only the new changes → **append** |
| Silver | rebuild from all Bronze rows → **overwrite** | only new Bronze rows → **MERGE** |
| Gold | one version per partner → **overwrite** (history restarts) | only changed partners → **SCD2 MERGE** |
| Use it for | first load, recovery, re-baselining | every scheduled run |

The very first `incremental` run also works: the tables are empty, so everything counts as new.
A `full` load rebuilds from the **current** state: partners already deleted in Lakebase are not included (their old
deliveries no longer find a partner) and Gold history restarts at version 1 — that is why `incremental` is the normal mode.

## 1. Configuration & Imports

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, when, lit, trim, lower, upper, initcap,
    to_timestamp, to_date, datediff, current_timestamp,
    regexp_replace, concat_ws, sha2, row_number, explode,
    coalesce, round as spark_round, expr, size, split,
    year, month, dayofweek, hour, date_format,
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## 2. Pipeline Parameters

In [ ]:
dbutils.widgets.text("env", "dev", "Environment")
dbutils.widgets.text("catalog_name", "zomato_analytics", "Unity Catalog name")

ENV = dbutils.widgets.get("env")
CATALOG = dbutils.widgets.get("catalog_name")

# Schema names within our dedicated catalog
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Fully qualified namespaces
BRONZE_FQN = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER_FQN = f"{CATALOG}.{SILVER_SCHEMA}"

print(f"Environment   : {ENV}")
print(f"Catalog       : {CATALOG}")
print(f"Bronze source : {BRONZE_FQN}")
print(f"Silver target : {SILVER_FQN}")

In [ ]:
# --- Lakebase source (Exercise 5) ---
dbutils.widgets.dropdown("enable_lakebase_cdc", "true", ["true", "false"], "Enable Lakebase source")
dbutils.widgets.dropdown("lakebase_load_type", "incremental", ["incremental", "full"], "Lakebase load type")

ENABLE_LAKEBASE_CDC = dbutils.widgets.get("enable_lakebase_cdc") == "true"
LAKEBASE_LOAD_TYPE = dbutils.widgets.get("lakebase_load_type")
print(f"Lakebase source : {'enabled' if ENABLE_LAKEBASE_CDC else 'disabled'} ({LAKEBASE_LOAD_TYPE} load) — silver.delivery_partners")

## 3. Create Silver Schema (Idempotent — No Impact on Existing Objects)

In [ ]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_FQN}")
spark.sql(f"COMMENT ON SCHEMA {SILVER_FQN} IS 'Silver layer — cleansed, deduplicated, and conformed data'")

print(f"✓ Schema '{SILVER_FQN}' ready")

## 3. Shared Transformation Utilities

In [ ]:
class DataQualityMetrics:
    """Track data quality metrics across transformations."""

    def __init__(self, table_name: str):
        self.table_name = table_name
        self.input_count = 0
        self.output_count = 0
        self.duplicates_removed = 0
        self.nulls_filled = 0
        self.invalid_records_quarantined = 0

    def summary(self) -> dict:
        return {
            "table": self.table_name,
            "input_rows": self.input_count,
            "output_rows": self.output_count,
            "duplicates_removed": self.duplicates_removed,
            "nulls_filled": self.nulls_filled,
            "quarantined": self.invalid_records_quarantined,
            # An incremental run with no new input has nothing to fail → 100%
            "pass_rate_pct": round(self.output_count / self.input_count * 100, 2) if self.input_count else 100.0,
        }


def deduplicate(df: DataFrame, key_columns: list, order_col: str = "_bronze_loaded_at") -> DataFrame:
    """Remove duplicates keeping the latest record per key."""
    window = Window.partitionBy(*key_columns).orderBy(col(order_col).desc())
    return df.withColumn("_rn", row_number().over(window)).filter(col("_rn") == 1).drop("_rn")


def standardize_text_columns(df: DataFrame, columns: list) -> DataFrame:
    """Trim whitespace and apply proper casing."""
    for c in columns:
        df = df.withColumn(c, initcap(trim(col(c))))
    return df

## 4. Silver — Customers

In [ ]:
def transform_customers() -> DataQualityMetrics:
    """Cleanse and transform customer data."""
    metrics = DataQualityMetrics("silver_customers")

    df = spark.table(f"{BRONZE_FQN}.customers")
    metrics.input_count = df.count()

    # Deduplicate
    df = deduplicate(df, ["customer_id"])
    metrics.duplicates_removed = metrics.input_count - df.count()

    # Cleanse & transform
    df_clean = (
        df
        .withColumn("first_name", initcap(trim(col("first_name"))))
        .withColumn("last_name", initcap(trim(col("last_name"))))
        .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
        .withColumn("email", lower(trim(col("email"))))
        .withColumn("phone_cleaned", regexp_replace(col("phone"), "[^0-9+]", ""))
        .withColumn("signup_date", to_timestamp(col("signup_date")))
        .withColumn("signup_year", year(col("signup_date")))
        .withColumn("signup_month", month(col("signup_date")))
        .withColumn("city", initcap(trim(col("city"))))
        .withColumn("subscription_tier", coalesce(col("subscription_tier"), lit("Free")))
        .withColumn("is_active", coalesce(col("is_active"), lit(True)))
        .withColumn("total_orders_lifetime", coalesce(col("total_orders_lifetime"), lit(0)))
        .withColumn("avg_order_value", coalesce(col("avg_order_value"), lit(0.0)))
        .withColumn(
            "customer_segment",
            when(col("total_orders_lifetime") >= 100, "Power User")
            .when(col("total_orders_lifetime") >= 30, "Regular")
            .when(col("total_orders_lifetime") >= 5, "Occasional")
            .otherwise("New"),
        )
        .withColumn("_silver_loaded_at", current_timestamp())
        .withColumn("_data_quality_score", lit(1.0))
        .drop("_ingested_at", "_source_system", "_source_file", "_row_hash", "_bronze_loaded_at")
    )

    valid_df = df_clean.filter(
        col("customer_id").isNotNull()
        & col("email").isNotNull()
        & col("email").rlike("^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z]{2,}$")
    )
    metrics.invalid_records_quarantined = df_clean.count() - valid_df.count()

    valid_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_FQN}.customers")

    metrics.output_count = valid_df.count()
    return metrics

## 5. Silver — Restaurants

In [ ]:
def transform_restaurants() -> DataQualityMetrics:
    """Cleanse and transform restaurant data."""
    metrics = DataQualityMetrics("silver_restaurants")

    df = spark.table(f"{BRONZE_FQN}.restaurants")
    metrics.input_count = df.count()

    df = deduplicate(df, ["restaurant_id"])
    metrics.duplicates_removed = metrics.input_count - df.count()

    df_clean = (
        df
        .withColumn("name", initcap(trim(col("name"))))
        .withColumn("city", initcap(trim(col("city"))))
        .withColumn("locality", initcap(trim(col("locality"))))
        .withColumn("cuisines_list", concat_ws(", ", col("cuisines")))
        .withColumn("num_cuisines", size(col("cuisines")))
        .withColumn("onboarded_date", to_timestamp(col("onboarded_date")))
        .withColumn(
            "price_tier",
            when(col("avg_cost_for_two") <= 300, "Budget")
            .when(col("avg_cost_for_two") <= 700, "Mid-Range")
            .when(col("avg_cost_for_two") <= 1500, "Premium")
            .otherwise("Luxury"),
        )
        .withColumn(
            "rating_tier",
            when(col("rating") >= 4.5, "Excellent")
            .when(col("rating") >= 4.0, "Very Good")
            .when(col("rating") >= 3.5, "Good")
            .when(col("rating") >= 3.0, "Average")
            .otherwise("Below Average"),
        )
        .withColumn("rating", spark_round(col("rating"), 1))
        .withColumn("is_delivering_now", coalesce(col("is_delivering_now"), lit(False)))
        .withColumn("has_online_delivery", coalesce(col("has_online_delivery"), lit(False)))
        .withColumn("has_table_booking", coalesce(col("has_table_booking"), lit(False)))
        .withColumn("is_premium_partner", coalesce(col("is_premium_partner"), lit(False)))
        .withColumn("_silver_loaded_at", current_timestamp())
        .drop("_ingested_at", "_source_system", "_source_file", "_row_hash", "_bronze_loaded_at")
    )

    valid_df = df_clean.filter(col("restaurant_id").isNotNull() & col("name").isNotNull())
    metrics.invalid_records_quarantined = df_clean.count() - valid_df.count()

    valid_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_FQN}.restaurants")
    metrics.output_count = valid_df.count()
    return metrics

## 6. Silver — Orders

In [ ]:
def transform_orders() -> DataQualityMetrics:
    """Cleanse and transform order data."""
    metrics = DataQualityMetrics("silver_orders")

    df = spark.table(f"{BRONZE_FQN}.orders")
    metrics.input_count = df.count()

    df = deduplicate(df, ["order_id"])
    metrics.duplicates_removed = metrics.input_count - df.count()

    df_clean = (
        df
        .withColumn("order_placed_at", to_timestamp(col("order_placed_at")))
        .withColumn("order_date", to_date(col("order_placed_at")))
        .withColumn("order_year", year(col("order_placed_at")))
        .withColumn("order_month", month(col("order_placed_at")))
        .withColumn("order_day_of_week", dayofweek(col("order_placed_at")))
        .withColumn("order_hour", hour(col("order_placed_at")))
        .withColumn("order_day_name", date_format(col("order_placed_at"), "EEEE"))
        .withColumn(
            "order_time_slot",
            when(col("order_hour").between(6, 10), "Breakfast")
            .when(col("order_hour").between(11, 14), "Lunch")
            .when(col("order_hour").between(15, 17), "Snacks")
            .when(col("order_hour").between(18, 22), "Dinner")
            .otherwise("Late Night"),
        )
        .withColumn("order_status", upper(trim(col("order_status"))))
        .withColumn("payment_status", upper(trim(col("payment_status"))))
        .withColumn("is_delivered", when(col("order_status") == "DELIVERED", True).otherwise(False))
        .withColumn("is_cancelled", when(col("order_status").isin("CANCELLED", "REFUNDED"), True).otherwise(False))
        .withColumn("has_discount", when(col("discount_pct") > 0, True).otherwise(False))
        .withColumn("total_amount", when(col("total_amount") < 0, lit(0.0)).otherwise(col("total_amount")))
        .withColumn("subtotal", spark_round(col("subtotal"), 2))
        .withColumn("total_amount", spark_round(col("total_amount"), 2))
        .withColumn("discount_amount", spark_round(col("discount_amount"), 2))
        .withColumn("tax_amount", spark_round(col("tax_amount"), 2))
        .withColumn("delivery_fee", spark_round(col("delivery_fee"), 2))
        .withColumn("_silver_loaded_at", current_timestamp())
        .drop("_ingested_at", "_source_system", "_source_file", "_row_hash", "_bronze_loaded_at")
    )

    # Referential integrity — only keep orders with valid customer & restaurant IDs
    valid_customers = spark.table(f"{SILVER_FQN}.customers").select("customer_id")
    valid_restaurants = spark.table(f"{SILVER_FQN}.restaurants").select("restaurant_id")

    valid_df = (
        df_clean
        .join(valid_customers, "customer_id", "inner")
        .join(valid_restaurants, "restaurant_id", "inner")
    )
    metrics.invalid_records_quarantined = df_clean.count() - valid_df.count()

    valid_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_FQN}.orders")
    metrics.output_count = valid_df.count()
    return metrics

## 7. Silver — Deliveries

In [ ]:
def transform_deliveries() -> DataQualityMetrics:
    """Cleanse and transform delivery data."""
    metrics = DataQualityMetrics("silver_deliveries")

    df = spark.table(f"{BRONZE_FQN}.deliveries")
    metrics.input_count = df.count()

    df = deduplicate(df, ["delivery_id"])
    metrics.duplicates_removed = metrics.input_count - df.count()

    df_clean = (
        df
        .withColumn("pickup_timestamp", to_timestamp(col("pickup_timestamp")))
        .withColumn("delivered_timestamp", to_timestamp(col("delivered_timestamp")))
        .withColumn("delivery_status", upper(trim(col("delivery_status"))))
        .withColumn("vehicle_type", initcap(trim(col("vehicle_type"))))
        .withColumn(
            "delivery_sla_status",
            when(col("delivery_time_mins") <= 30, "Within SLA")
            .when(col("delivery_time_mins") <= 45, "Near SLA")
            .otherwise("SLA Breached"),
        )
        .withColumn(
            "distance_bucket",
            when(col("delivery_distance_km") <= 3, "Short (0-3 km)")
            .when(col("delivery_distance_km") <= 7, "Medium (3-7 km)")
            .when(col("delivery_distance_km") <= 12, "Long (7-12 km)")
            .otherwise("Very Long (12+ km)"),
        )
        .withColumn("delivery_rating", when(col("delivery_rating").between(1, 5), col("delivery_rating")).otherwise(lit(None)))
        .withColumn("delivery_distance_km", spark_round(col("delivery_distance_km"), 2))
        .withColumn("_silver_loaded_at", current_timestamp())
        .drop("_ingested_at", "_source_system", "_source_file", "_row_hash", "_bronze_loaded_at")
    )

    # Referential integrity check against orders
    valid_orders = spark.table(f"{SILVER_FQN}.orders").select("order_id")
    valid_df = df_clean.join(valid_orders, "order_id", "inner")
    metrics.invalid_records_quarantined = df_clean.count() - valid_df.count()

    valid_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_FQN}.deliveries")
    metrics.output_count = valid_df.count()
    return metrics

## 8. Silver — Reviews

In [ ]:
def transform_reviews() -> DataQualityMetrics:
    """Cleanse and transform review data."""
    metrics = DataQualityMetrics("silver_reviews")

    df = spark.table(f"{BRONZE_FQN}.reviews")
    metrics.input_count = df.count()

    df = deduplicate(df, ["review_id"])
    metrics.duplicates_removed = metrics.input_count - df.count()

    df_clean = (
        df
        .withColumn("review_timestamp", to_timestamp(col("review_timestamp")))
        .withColumn("review_date", to_date(col("review_timestamp")))
        .withColumn("rating", when(col("rating").between(1, 5), col("rating")).otherwise(lit(None)))
        .withColumn("review_text", trim(col("review_text")))
        .withColumn("review_text_length", when(col("review_text").isNotNull(), expr("length(review_text)")).otherwise(lit(0)))
        .withColumn("sentiment_label", initcap(trim(col("sentiment_label"))))
        .withColumn("_silver_loaded_at", current_timestamp())
        .drop("_ingested_at", "_source_system", "_source_file", "_row_hash", "_bronze_loaded_at")
    )

    valid_df = df_clean.filter(col("review_id").isNotNull() & col("rating").isNotNull())
    metrics.invalid_records_quarantined = df_clean.count() - valid_df.count()

    valid_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER_FQN}.reviews")
    metrics.output_count = valid_df.count()
    return metrics

## 8A. Silver — Delivery Partners from Lakebase CDC (Exercise 5)

Steps 1 → 4 of the Silver algorithm at the top of this notebook.

In [ ]:
from pyspark.sql.functions import max as spark_max

BRONZE_PARTNERS_TABLE = f"{BRONZE_FQN}.lakebase_delivery_partners_cdc"
SILVER_PARTNERS_TABLE = f"{SILVER_FQN}.delivery_partners"
SILVER_PARTNER_COLUMNS = [
    "partner_id", "partner_name", "phone", "phone_cleaned", "email", "city", "vehicle_type",
    "partner_status", "shift_preference", "rating", "partner_tier", "joined_date", "kyc_verified",
    "is_active", "is_deleted", "deleted_at", "source_created_at", "source_updated_at",
    "_cdc_last_op", "_cdc_change_seq", "_cdc_txid", "_cdc_changed_at", "_silver_loaded_at",
]


def read_partner_rows_from_bronze():
    """Step 1 — FULL: every Bronze row. INCREMENTAL: only Bronze rows not applied to Silver yet."""
    bronze_df = spark.table(BRONZE_PARTNERS_TABLE)
    if LAKEBASE_LOAD_TYPE == "full":
        return bronze_df
    last_txid = spark.table(SILVER_PARTNERS_TABLE).agg(spark_max("_cdc_txid")).first()[0]  # Silver bookmark
    if last_txid is None:
        return bronze_df  # Silver is still empty → everything is new
    return bronze_df.filter(col("_cdc_txid") > last_txid)


def keep_newest_change_per_partner(rows_df):
    """Step 2 — if a partner changed several times, keep only the newest change."""
    newest_first = Window.partitionBy("partner_id").orderBy(col("_cdc_change_seq").desc(), col("_bronze_loaded_at").desc())
    return (
        rows_df
        .withColumn("_rank", row_number().over(newest_first))
        .filter(col("_rank") == 1)
        .drop("_rank")
    )


def clean_partners(newest_df):
    """Step 3 — standardize values like every other Silver table; DELETE becomes a soft delete."""
    is_deleted = col("_cdc_op") == "DELETE"
    return (
        newest_df
        .withColumn("partner_name", initcap(trim(col("partner_name"))))
        .withColumn("phone_cleaned", regexp_replace(col("phone"), "[^0-9+]", ""))
        .withColumn("email", lower(trim(col("email"))))
        .withColumn("city", initcap(trim(col("city"))))
        .withColumn("vehicle_type", initcap(trim(col("vehicle_type"))))
        .withColumn("partner_status", upper(trim(col("partner_status"))))
        .withColumn("shift_preference", initcap(trim(col("shift_preference"))))
        .withColumn("rating", when(col("rating").between(1, 5), spark_round(col("rating"), 2)))  # invalid → NULL
        .withColumn(
            "partner_tier",
            when(col("rating") >= 4.5, "Elite")
            .when(col("rating") >= 4.0, "Preferred")
            .when(col("rating") >= 3.0, "Standard")
            .when(col("rating").isNotNull(), "Probation")
            .otherwise("Unrated"),
        )
        .withColumn("joined_date", to_date(col("joined_date")))
        .withColumn("kyc_verified", coalesce(col("kyc_verified"), lit(False)))
        .withColumn("is_deleted", is_deleted)
        .withColumn("deleted_at", when(is_deleted, col("_cdc_changed_at")))
        .withColumn("is_active", (col("partner_status") == "ACTIVE") & ~is_deleted)
        .withColumn("source_created_at", to_timestamp(col("created_at")))
        .withColumn("source_updated_at", to_timestamp(col("updated_at")))
        .withColumnRenamed("_cdc_op", "_cdc_last_op")
        .withColumn("_silver_loaded_at", current_timestamp())
        .select(*SILVER_PARTNER_COLUMNS)
    )


# Step 4 (incremental) — insert new partners, update known partners only with NEWER changes
MERGE_PARTNERS_SQL = f"""
    MERGE INTO {SILVER_PARTNERS_TABLE} AS target
    USING partner_updates AS source
    ON target.partner_id = source.partner_id
    WHEN MATCHED AND source._cdc_change_seq > target._cdc_change_seq THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
"""


def transform_delivery_partners() -> DataQualityMetrics:
    """Lakebase CDC → silver.delivery_partners (full: rebuild, incremental: MERGE)."""
    metrics = DataQualityMetrics("silver_delivery_partners")

    # Step 1 — which Bronze rows do we need?
    rows = read_partner_rows_from_bronze()
    metrics.input_count = rows.count()
    print(f"  {LAKEBASE_LOAD_TYPE.upper()} load: {metrics.input_count:,} Bronze rows to process")
    if metrics.input_count == 0:
        return metrics

    # Drop broken rows (no partner id or unknown operation)
    valid_rows = rows.filter(col("partner_id").isNotNull() & col("_cdc_op").isin("SNAPSHOT", "INSERT", "UPDATE", "DELETE"))
    valid_count = valid_rows.count()
    metrics.invalid_records_quarantined = metrics.input_count - valid_count

    # Steps 2 + 3 — newest change per partner, cleaned
    partners = clean_partners(keep_newest_change_per_partner(valid_rows))
    metrics.duplicates_removed = valid_count - partners.count()  # older changes replaced by a newer one

    # Step 4 — write
    if LAKEBASE_LOAD_TYPE == "full":
        partners.write.format("delta").mode("overwrite").saveAsTable(SILVER_PARTNERS_TABLE)
        print(f"  ✓ {SILVER_PARTNERS_TABLE} rebuilt (overwrite)")
    else:
        partners.createOrReplaceTempView("partner_updates")
        spark.sql(MERGE_PARTNERS_SQL)
        print(f"  ✓ {SILVER_PARTNERS_TABLE} merged — older/duplicate changes replaced: {metrics.duplicates_removed:,}")

    metrics.output_count = valid_count  # pass rate = valid rows / received rows
    return metrics

## 9. Execute Silver Transformation Pipeline

In [ ]:
import json

transform_functions = [
    transform_customers,
    transform_restaurants,
    transform_orders,
    transform_deliveries,
    transform_reviews,
]

# Exercise 5 — delivery partners from Lakebase CDC (full or incremental)
if ENABLE_LAKEBASE_CDC:
    transform_functions.append(transform_delivery_partners)

all_metrics = []
for func in transform_functions:
    print(f"\n{'='*50}")
    print(f"Processing: {func.__name__}")
    print(f"{'='*50}")
    metrics = func()
    all_metrics.append(metrics.summary())
    print(json.dumps(metrics.summary(), indent=2))

## 10. Silver Layer Validation Summary

In [ ]:
print("\n" + "=" * 70)
print(f"SILVER LAYER — TRANSFORMATION SUMMARY ({SILVER_FQN})")
print("=" * 70)

for m in all_metrics:
    status = "PASS" if m["pass_rate_pct"] >= 95 else "WARN" if m["pass_rate_pct"] >= 80 else "FAIL"
    print(f"  [{status}] {m['table']}: {m['input_rows']:,} → {m['output_rows']:,} "
          f"(dedup: {m['duplicates_removed']:,}, quarantined: {m['quarantined']:,}, "
          f"pass: {m['pass_rate_pct']}%)")

print("=" * 70)
print(f"🟢 Silver transformation pipeline completed successfully.")
print(f"   All tables created under: {SILVER_FQN}.*")

## 11. Verify Isolation — List Only Our Objects

In [ ]:
print(f"Tables in {SILVER_FQN}:")
spark.sql(f"SHOW TABLES IN {SILVER_FQN}").show(truncate=False)